# Phase 6 — Calibration + Evaluation  (completes the core pipeline)

**Why (paper §3.8, §3.11):** the trained model's three outputs are *ordered* but not yet
*guaranteed* to contain the truth 90% of the time. **Conformal prediction** earns that
guarantee using the held-out calibration set:

1. On calibration, measure how far each truth fell outside the predicted range.
2. Take the 90% mark of those misses → a single number **Q**.
3. Widen every interval by **Q** (clip the lower end at 0).

Then we report the full metrics — all in **AQI points** — and, if you trained more than one
split strategy, compare them side by side to expose the leakage gap.

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST (set REPO_URL to your repo) ===
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os, sys, subprocess

def _find_repo_root():
    # Are we already inside the repo (or just above the notebooks/ folder)?
    for cand in (".", "..", "pm25-visual-aq"):
        if os.path.isdir(os.path.join(cand, "src")):
            return os.path.abspath(cand)
    return None

_root = _find_repo_root()
if _root is None:                       # fresh Colab session: clone the code
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pm25-visual-aq"], check=True)
    _root = os.path.abspath("pm25-visual-aq")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

print("repo root:", _root, "| Colab:", IN_COLAB)

In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt
from src.config import load_config
from src import data, splits, physics, dataset as D, model as M, train as T
from src import calibrate as C, metrics as Mx
cfg = load_config()
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

SOURCE = cfg["data"]["drive_path"]        # laptop test: "tests/fixture_ds"
ds, df = data.load_clean(SOURCE, from_disk=True, seed=cfg["seed"])
cache = physics.load_map_cache(os.path.join(cfg["data"]["cache_dir"], "physics_maps_%d.npy" % cfg["data"]["image_size"]))
out_root = cfg["data"]["outputs_dir"]; os.makedirs(out_root, exist_ok=True)
print("device:", device)

## Evaluate a trained model

`evaluate_strategy` rebuilds that split, loads its checkpoint, predicts on calibration and
test, computes **Q** from calibration, applies it to test, and returns the metrics. It also
saves **Q** next to the checkpoint so the demo (Phase 10) can reuse it.

In [ ]:
def evaluate_strategy(strategy):
    sp = splits.make_splits(df, strategy=strategy, seed=cfg["seed"],
            station_col=cfg["data"]["station_col"], time_col=cfg["data"]["time_col"],
            lon_col=cfg["data"]["lon_col"], lat_col=cfg["data"]["lat_col"])
    loaders = D.make_dataloaders(ds, sp, cache, cfg, num_workers=2)
    net = M.build_model(cfg).to(device)
    T.load_checkpoint(os.path.join(out_root, strategy, "best_model.pth"), net, map_location=device)
    lg = cfg["train"]["log_target"]
    cal_p, cal_y = T.collect_predictions(net, loaders["cal"], device, lg)
    test_p, test_y = T.collect_predictions(net, loaders["test"], device, lg)
    Q = C.conformal_Q(cal_p, cal_y, cfg["calibration"]["coverage"])
    json.dump({"Q": Q}, open(os.path.join(out_root, strategy, "conformal_Q.json"), "w"))
    cal_test = C.apply_conformal(test_p, Q)
    return {"strategy": strategy, "Q": Q,
            "raw_coverage": C.coverage(test_p, test_y),
            "report": Mx.full_report(cal_test, test_y, cfg["calibration"]["coverage"]),
            "preds": cal_test, "y": test_y}

primary = evaluate_strategy(cfg["split"]["strategy"])
print("strategy:", primary["strategy"], "| conformal Q = %.1f AQI" % primary["Q"])
print("coverage: raw %.3f -> calibrated %.3f (target %.2f)"
      % (primary["raw_coverage"], primary["report"]["coverage"], cfg["calibration"]["coverage"]))
{k: round(v,3) for k,v in primary["report"].items()}

## Where is the model weak?
Errors broken down by true-AQI band — the visual signal is weakest at low pollution.

In [ ]:
ebm = Mx.error_by_magnitude(primary["y"], primary["preds"][:,1])
display(ebm)
plt.figure(figsize=(7,3)); plt.bar(ebm["band"], ebm["MAE"]); plt.ylabel("MAE (AQI)"); plt.xlabel("true AQI band")
plt.title("Error by pollution level"); plt.show()

plt.figure(figsize=(7,3)); plt.hist(primary["preds"][:,2]-primary["preds"][:,0], bins=40)
plt.xlabel("interval width (AQI)"); plt.ylabel("count"); plt.title("Calibrated interval widths"); plt.show()

## A few example predictions

Each point is a test photo: the dot is the predicted median, the bar is the calibrated
90% interval, and the ✕ is the truth. Most truths should sit inside their bars.

In [ ]:
import numpy as np
idx = np.argsort(primary["y"])[::max(1, len(primary["y"])//40)][:40]
p, y = primary["preds"][idx], primary["y"][idx]
xs = np.arange(len(idx))
plt.figure(figsize=(9,4))
plt.vlines(xs, p[:,0], p[:,2], color="#4C78A8", lw=3, alpha=0.5, label="90% interval")
plt.plot(xs, p[:,1], "o", ms=4, color="#4C78A8", label="median")
plt.plot(xs, y, "x", ms=6, color="#E45756", label="truth")
plt.legend(); plt.xlabel("test photos (sorted by true AQI)"); plt.ylabel("AQI"); plt.title("Predictions vs truth"); plt.show()

## Leakage gap: compare every split you trained

If you trained more than one split strategy (e.g. re-ran `05_train` with
`split.strategy: random`), this tabulates them side by side. **`random` should look better
than `station_grouped`** — that difference is leakage inflation, measured not assumed.

In [ ]:
import pandas as pd
avail = [s for s in ["station_grouped","random","geographic","temporal"]
         if os.path.exists(os.path.join(out_root, s, "best_model.pth"))]
rows = []
for s in avail:
    r = evaluate_strategy(s)
    rows.append({"strategy": s, **{k: round(v,3) for k,v in r["report"].items()}})
table = pd.DataFrame(rows).set_index("strategy")
table.to_csv(os.path.join(out_root, "results_by_split.csv"))
table

## Core pipeline complete ✓

You now have: leakage-safe evaluation, a physics-guided model, and **calibrated intervals
with ~90% coverage**, all in AQI points. Paste me this notebook's numbers and I'll sanity-check
them and write them into `docs/RESULTS.md`.

**Next (contributions):** `07_error_ceiling` (C2 — how much error is unavoidable) and
`08_abstention` (C3 — refusing to answer on unusable inputs).